# Prompt Engineering Fundamentals

This notebook covers the practical foundation for prompt engineering in LangSmith and LangGraph:

- Why prompt engineering matters
- Prompt vs. prompt template
- F-string and Mustache template formats
- A basic prompt-engineering workflow
- Prompt chaining with LangGraph
- Managing prompts in LangSmith

The structure follows the official LangChain docs for prompt engineering concepts, prompt template format, the quickstart, prompt chaining, and prompt management.

## Learning goals

By the end of this notebook, you should be able to:

1. Explain why prompts shape model behavior.
2. Choose between chat prompts and completion prompts.
3. Build reusable prompt templates with variables.
4. Recognize when to use f-string vs. Mustache templates.
5. Sketch a prompt-engineering workflow for testing and iteration.
6. Understand prompt chaining as a multi-step LLM workflow.
7. Manage prompts as versioned assets in LangSmith.

## 1) Why prompt engineering matters

A prompt guides the model’s behavior without changing the model itself. In practice, prompt engineering helps you steer output quality, format, tone, and task performance with relatively low friction compared with heavier approaches.

### What prompts can control

- Role and tone
- Output format
- Degree of detail
- Task boundaries
- Use of examples
- Safety and refusal behavior

In [ ]:
# No external dependencies are required for the conceptual examples below.
# If you want to connect this notebook to LangSmith or LangGraph later,
# install the relevant packages in your environment.

## 2) Prompt types

LangSmith distinguishes between two broad prompt styles:

- **Chat prompts**: a list of messages with roles like system, user, and assistant.
- **Completion prompts**: a single text string.

For new projects, chat prompts are generally the recommended format because they map cleanly to modern chat model APIs and multi-turn interactions.

In [1]:
# Example: conceptual chat prompt structure

chat_prompt = [
    {"role": "system", "content": "You are a helpful tutor."},
    {"role": "user", "content": "Explain prompt engineering in one paragraph."},
]

chat_prompt

[{'role': 'system', 'content': 'You are a helpful tutor.'},
 {'role': 'user', 'content': 'Explain prompt engineering in one paragraph.'}]

## 3) Prompt vs. prompt template

A **prompt** is the message or messages sent to the model.

A **prompt template** is a reusable version of that prompt with variables that get filled in at runtime.

This is useful when you want:
- reusability
- testing with different inputs
- collaboration across teams
- versioned prompt assets

In [2]:
# Example: prompt template idea

template = """You are a customer support agent.

Policy:
{policy}

Question:
{question}
"""

inputs = {
    "policy": "Refunds are allowed within 7 days.",
    "question": "Can I get a refund for my purchase?"
}

filled_prompt = template.format(**inputs)
print(filled_prompt)

You are a customer support agent.

Policy:
Refunds are allowed within 7 days.

Question:
Can I get a refund for my purchase?



## 4) Prompt template formats

LangSmith supports two prompt template formats:

### f-string
Use single braces like `{variable}` for simple substitution.

Best for:
- flat inputs
- straightforward prompts
- simple variable injection

### Mustache
Use double braces like `{{variable}}` for more complex structures.

Best for:
- nested data
- loops
- conditionals
- evaluator workflows

In [3]:
# f-string style example

f_string_template = "Hello, {name}. Summarize {topic} in 3 bullets."
filled = f_string_template.format(name="Asha", topic="vector databases")
print(filled)

Hello, Asha. Summarize vector databases in 3 bullets.


In [4]:
# Mustache-style example (conceptual)
# In a real LangSmith prompt template, Mustache is useful when you need
# nested data, loops, or conditionals.

mustache_template = """
Hello, {{user.name}}.

Tasks:
{{#tasks}}
- {{.}}
{{/tasks}}
"""

print(mustache_template)


Hello, {{user.name}}.

Tasks:
{{#tasks}}
- {{.}}
{{/tasks}}



## 5) Quick workflow for prompt engineering

A practical prompt workflow usually looks like this:

1. Start with a clear task.
2. Draft a prompt with explicit instructions.
3. Add variables for dynamic inputs.
4. Test against several examples.
5. Compare outputs.
6. Refine the wording, structure, and constraints.
7. Version the winning prompt.

In [5]:
def build_prompt(question: str, style: str = "concise") -> str:
    return f"""You are a helpful assistant.

Answer style: {style}
Question: {question}
"""

print(build_prompt("What is prompt engineering?", style="concise"))

You are a helpful assistant.

Answer style: concise
Question: What is prompt engineering?



## 6) Prompt engineering quickstart mindset

The LangSmith quickstart frames prompt engineering as a cycle of creating, testing, and improving prompts using either the UI or the SDK. The important habit is to treat prompts as editable assets rather than one-off strings.

### Good prompt habits

- State the role clearly.
- Define the task explicitly.
- Specify the expected output shape.
- Add examples when helpful.
- Keep constraints visible.
- Test on real inputs.

In [6]:
# Example: improve a vague prompt

vague = "Tell me about AWS."

better = """You are a cloud tutor.
Explain AWS in simple language for a beginner.
Limit the answer to 5 bullet points.
"""

print(better)

You are a cloud tutor.
Explain AWS in simple language for a beginner.
Limit the answer to 5 bullet points.



## 7) Prompt chaining in LangGraph

Prompt chaining means each LLM call processes the output of the previous call. This is useful when a task can be broken into smaller steps, such as:

- draft → refine → polish
- extract → verify → summarize
- translate → review → format

In [7]:
# Conceptual prompt chaining example

draft = "Write a short product description for a smart lamp."
refine = f"Improve clarity and marketing appeal: {draft}"
polish = f"Make it sound premium and concise: {refine}"

print("Draft:", draft)
print("Refine:", refine)
print("Polish:", polish)

Draft: Write a short product description for a smart lamp.
Refine: Improve clarity and marketing appeal: Write a short product description for a smart lamp.
Polish: Make it sound premium and concise: Improve clarity and marketing appeal: Write a short product description for a smart lamp.


### Why chaining helps

- Easier debugging
- Better control over intermediate outputs
- Smaller, verifiable steps
- Cleaner separation of responsibilities

In [8]:
# Simple chaining pattern you can reuse in notebooks

def step_1(topic: str) -> str:
    return f"Draft a short explanation about {topic}."

def step_2(text: str) -> str:
    return f"Rewrite the following more clearly: {text}"

def step_3(text: str) -> str:
    return f"Add one practical example to: {text}"

draft = step_1("prompt templates")
refined = step_2(draft)
final = step_3(refined)

print(final)

Add one practical example to: Rewrite the following more clearly: Draft a short explanation about prompt templates.


## 8) Managing prompts in LangSmith

LangSmith lets you store, version, and manage prompts as shared assets. That makes prompts easier to review, tag, promote, and reuse across environments.

This matters when:
- multiple people edit prompts
- you need version history
- you want to test prompt changes safely
- you need a promotion flow from staging to production

### Prompt management concepts

- **Commits**: saved versions of a prompt
- **Tags**: labels that point to a specific version
- **Owners / permissions**: control who can edit or promote prompts
- **Webhooks**: trigger automation when prompts change

In [9]:
# Conceptual prompt registry structure

prompt_registry = {
    "support_classifier": {
        "version": "v3",
        "format": "f-string",
        "status": "production",
    },
    "summary_prompt": {
        "version": "v5",
        "format": "mustache",
        "status": "staging",
    },
}

prompt_registry

{'support_classifier': {'version': 'v3',
  'format': 'f-string',
  'status': 'production'},
 'summary_prompt': {'version': 'v5',
  'format': 'mustache',
  'status': 'staging'}}

## 9) When to use which format

### Use a chat prompt when:
- you are working with a conversational model
- you have multiple turns
- you want system/user/assistant separation

### Use an f-string template when:
- the prompt is simple
- your variables are flat
- you only need direct substitution

### Use a Mustache template when:
- your input is structured
- you need loops or conditionals
- you are building more advanced prompt logic

In [10]:
def choose_template_format(data_complexity: str) -> str:
    if data_complexity == "simple":
        return "f-string"
    if data_complexity == "structured":
        return "mustache"
    return "chat prompt"

for item in ["simple", "structured", "conversation"]:
    print(item, "->", choose_template_format(item))

simple -> f-string
structured -> mustache
conversation -> chat prompt


## 10) Mini exercise

Try this exercise in your own notebook or editor:

1. Rewrite a vague prompt into a specific prompt.
2. Add variables for the user input and output constraints.
3. Decide whether the prompt should be chat, f-string, or Mustache.
4. Split a complex task into a 2-step or 3-step chain.
5. Think about how you would version the best prompt in LangSmith.

## Key takeaways

- Prompt engineering shapes model behavior without changing the model.
- Chat prompts are the default choice for most new applications.
- Prompt templates make prompts reusable and testable.
- f-string templates are best for simple substitution.
- Mustache templates are better for nested and conditional structures.
- Prompt chaining is useful when a task can be broken into steps.
- LangSmith helps you version and manage prompts as assets.

## References

- https://docs.langchain.com/langsmith/prompt-engineering-concepts 
- https://docs.langchain.com/langsmith/prompt-template-format 
- https://docs.langchain.com/langsmith/prompt-engineering-quickstart 
- https://docs.langchain.com/oss/python/langgraph/workflows-agents#prompt-chaining 
- https://docs.langchain.com/langsmith/manage-prompts